# Yatri AI — Kumbh Mela 2027 Model Training

Fine-tunes Qwen2.5-7B on 3,366 Kumbh Mela QA pairs using QLoRA.

**Requirements:** T4 GPU (free Colab) or better.

**Steps:**
1. Upload `colab_training.zip`
2. Run all cells
3. Download the GGUF file
4. Place in `~/Kumbh/models/kumbh_model_q4_k_m.gguf` on your machine

In [ ]:
# Cell 1 — Fresh start with 3B model (fits T4 perfectly)
import torch, gc
gc.collect()
torch.cuda.empty_cache()

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo hf_transfer tyro xformers
!pip install -q "datasets>=3.4.1,<4.4.0" "transformers>=4.51.3,<=5.5.0,!=5.0.0" "trl>=0.18.2,<=0.24.0"

from unsloth import FastLanguageModel

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # 3B fits T4 easily
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

model.print_trainable_parameters()
print(f'\nModel loaded: {MODEL_NAME}')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB / 15 GB')

# Cell 2 — Upload and prepare data
from google.colab import files
import json
from datasets import Dataset

print('Upload colab_training.zip...')
uploaded = files.upload()
!unzip -o colab_training.zip

DATA_PATH = 'all_languages_combined.jsonl'
records = [json.loads(l) for l in open(DATA_PATH) if l.strip()]
print(f'Loaded {len(records)} QA pairs')

SYSTEM_MSG = 'You are Yatri AI, a helpful multilingual assistant for Nashik Kumbh Mela 2027. Answer accurately and concisely.'

def format_chat(rec):
    messages = [
        {'role': 'system', 'content': SYSTEM_MSG},
        {'role': 'user', 'content': rec['instruction']},
        {'role': 'assistant', 'content': rec['output']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(records).map(format_chat)
split = dataset.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(split["train"])}, Eval: {len(split["test"])}')

# Cell 3 — Train (~20 min on T4)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        output_dir='/content/kumbh_model',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_steps=30,
        logging_steps=10,
        save_steps=200,
        fp16=True,
        optim='paged_adamw_8bit',
        report_to='none',
        save_total_limit=2,
    ),
)

print('Training...')
trainer.train()
trainer.save_model('/content/kumbh_model')
tokenizer.save_pretrained('/content/kumbh_model')
print('Done!')

# Cell 4 — Test
FastLanguageModel.for_inference(model)

for q in ['When is Kumbh Mela 2027?', 'नाशिक में कुंभ मेला कब है?', 'How to reach Trimbakeshwar?']:
    msgs = [{'role':'system','content':SYSTEM_MSG}, {'role':'user','content':q}]
    inp = tokenizer.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to('cuda')
    out = model.generate(inp, max_new_tokens=150, temperature=0.3, do_sample=True)
    print(f'Q: {q}\nA: {tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True)}\n')

# Cell 5 — Export GGUF and download
model.save_pretrained_gguf('/content/kumbh_gguf', tokenizer, quantization_method='q4_k_m')

import shutil
from google.colab import files
shutil.copy('/content/kumbh_gguf/unsloth.Q4_K_M.gguf', '/content/kumbh_model_q4_k_m.gguf')
print('Downloading GGUF (~2GB)...')
files.download('/content/kumbh_model_q4_k_m.gguf')

In [ ]:
import os, glob, shutil
from google.colab import files

# Find the actual GGUF file
gguf_files = glob.glob('/content/kumbh_gguf/*.gguf') + glob.glob('/content/kumbh_gguf/**/*.gguf')
print("Found GGUF files:", gguf_files)

if not gguf_files:
    # Maybe it's in a different location
    gguf_files = glob.glob('/content/**/*.gguf', recursive=True)
    print("All GGUF files on disk:", gguf_files)

if gguf_files:
    src = gguf_files[0]
    print(f"Downloading: {src} ({os.path.getsize(src)/1e9:.1f} GB)")
    shutil.copy(src, '/content/kumbh_model_q4_k_m.gguf')
    files.download('/content/kumbh_model_q4_k_m.gguf')
else:
    print("No GGUF found. Listing /content/kumbh_gguf/:")
    os.listdir('/content/kumbh_gguf/')